<a href="https://colab.research.google.com/github/Kaustubh0505/C_G15_dvaCapstone/blob/main/notebooks/05_final_load_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Final Data Preparation for Tableau

#### This notebook prepares the cleaned dataset for visualization in Tableau.
#### It focuses on:
#### - Removing bias from imputed values
#### - Creating business-friendly features
#### - Generating KPIs
#### - Preparing aggregated datasets
#### - Exporting final files for dashboarding

In [52]:
import pandas as pd
import numpy as np

In [53]:
df = pd.read_csv("cleaned_data.csv")

In [54]:
print("Shape:", df.shape)
df.head()

Shape: (51717, 11)


,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_costfor_two_people,listed_intype,listed_incity
0,Jalsa,1,1,4.1,775,Banashankari,Casual Dining,North Indian,800.0,Buffet,Banashankari
1,Spice Elephant,1,0,4.1,787,Banashankari,Casual Dining,Chinese,800.0,Buffet,Banashankari
2,San Churro Cafe,1,0,3.8,918,Banashankari,Cafe,Cafe,800.0,Buffet,Banashankari
3,Addhuri Udupi Bhojana,0,0,3.7,88,Banashankari,Quick Bites,South Indian,300.0,Buffet,Banashankari
4,Grand Village,0,0,3.8,166,Basavanagudi,Casual Dining,North Indian,600.0,Buffet,Banashankari


In [55]:
IMPUTED_RATE = 3.700448817952718

df = df[df["rate"] != IMPUTED_RATE].copy()

print("Shape after removing imputed values:", df.shape)

Shape after removing imputed values: (41665, 11)


In [56]:
df["rating_category"] = pd.cut(
    df["rate"],
    bins=[0, 3, 4, 5],
    labels=["Low", "Medium", "High"]
)

In [57]:
df["cost_category"] = pd.cut(
    df["approx_costfor_two_people"],
    bins=[0, 300, 700, 6000],
    labels=["Budget", "Mid-range", "Premium"]
)

In [58]:
df["votes_category"] = pd.cut(
    df["votes"],
    bins=[-1, 50, 300, 20000],
    labels=["Low", "Medium", "High"]
)

In [59]:
df["is_premium"] = df["approx_costfor_two_people"].apply(lambda x: 1 if x > 700 else 0)

In [60]:
df["value_for_money"] = df["rate"] / df["approx_costfor_two_people"].replace(0, np.nan)

In [61]:
kpi = pd.DataFrame({
    "Metric": [
        "Total Restaurants",
        "Average Rating",
        "Average Cost",
        "Online Order %",
        "Table Booking %",
        "Premium Restaurants %"
    ],
    "Value": [
        df.shape[0],
        round(df["rate"].mean(), 2),
        round(df["approx_costfor_two_people"].mean(), 2),
        round(df["online_order"].mean() * 100, 2),
        round(df["book_table"].mean() * 100, 2),
        round(df["is_premium"].mean() * 100, 2)
    ]
})

kpi

,Metric,Value
0,Total Restaurants,41665.00
1,Average Rating,3.70
2,Average Cost,602.06
3,Online Order %,65.30
4,Table Booking %,15.13
5,Premium Restaurants %,24.84


In [62]:
top_locations = df["location"].value_counts().head(10).index

location_analysis = (
    df[df["location"].isin(top_locations)]
    .groupby("location")
    .agg(
        total_restaurants=("name", "count"),
        avg_rating=("rate", "mean"),
        avg_cost=("approx_costfor_two_people", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values(by="total_restaurants", ascending=False)
)

location_analysis

,location,total_restaurants,avg_rating,avg_cost
0,BTM,3930,3.57,418.77
6,Koramangala 5th Block,2319,4.01,678.05
2,HSR,2019,3.67,498.97
3,Indiranagar,1847,3.83,672.36
4,JP Nagar,1717,3.68,554.48
5,Jayanagar,1643,3.78,499.54
9,Whitefield,1582,3.62,677.18
8,Marathahalli,1443,3.54,557.14
1,Bannerghatta Road,1235,3.51,475.99
7,Koramangala 6th Block,1077,3.78,631.48


In [63]:
top_cuisines = df["cuisines"].value_counts().head(10).index

cuisine_analysis = (
    df[df["cuisines"].isin(top_cuisines)]
    .groupby("cuisines")
    .agg(
        total_restaurants=("name", "count"),
        avg_rating=("rate", "mean")
    )
    .round(2)
    .reset_index()
    .sort_values(by="total_restaurants", ascending=False)
)

In [64]:
rest_type_analysis = (
    df.groupby("rest_type")
    .agg(
        total_restaurants=("name", "count"),
        avg_rating=("rate", "mean")
    )
    .round(2)
    .reset_index()
)

In [65]:
service_analysis = (
    df.groupby(["online_order", "book_table"])
    .agg(
        avg_rating=("rate", "mean"),
        avg_votes=("votes", "mean")
    )
    .round(2)
    .reset_index()
)

In [66]:
df["restaurant_segment"] = pd.cut(
    df["approx_costfor_two_people"],
    bins=[0, 300, 700, 6000],
    labels=["Budget", "Mid-range", "Premium"]
)

In [67]:
df["popularity_segment"] = pd.cut(
    df["votes"],
    bins=[-1, 50, 300, 20000],
    labels=["Low", "Medium", "High"]
)

In [68]:
df.columns = [col.replace("_", " ").title() for col in df.columns]

In [69]:
df.to_csv("final_dataset.csv", index=False)

location_analysis.to_csv("location_analysis.csv", index=False)

cuisine_analysis.to_csv("cuisine_analysis.csv", index=False)

rest_type_analysis.to_csv("rest_type_analysis.csv", index=False)

service_analysis.to_csv("service_analysis.csv", index=False)

kpi.to_csv("kpi_summary.csv", index=False)

print("All files exported successfully")

All files exported successfully


## Summary
- Removed imputed ratings to reduce bias  
- Created business features (categories, premium, value)  
- Generated KPIs for dashboard use  
- Built aggregated datasets for visualization  
- Added simple segmentation  
- Exported final files for Tableau  